# Logprob-Reconstruction Contrastive Training (Mechanism F)

Trains a `LogprobReconProgressiveCompressor` using the combined objective:

$$\mathcal{L} = \mathcal{L}_{\text{SupCon}}(z) + \lambda \cdot \mathcal{L}_{\text{recon}}(g(z),\, \boldsymbol{\ell})$$

where $\boldsymbol{\ell}$ is the per-token logprob sequence and $g$ is a lightweight
MLP decoder that is **discarded at inference** — `forward()` is identical to
`ProgressiveCompressor`.  The auxiliary decoder forces the encoder to preserve
uncertainty-relevant dimensions (predictive coding regularizer).

**Prerequisite:** activations must have been logged with `--log-response-logprobs`
so that `response_token_logprobs` is stored in the zarr/LMDB store.

In [ ]:
import os
import sys
import glob
import re

import torch
import pandas as pd
from torch.utils.data import DataLoader
from loguru import logger
from tqdm.auto import tqdm

from activation_logging.activation_parser import ActivationParser
from activation_research.model import LogprobReconProgressiveCompressor
from activation_research.training import (
    train_contrastive_logprob_recon,
    _contrastive_collate_with_logprob,
)
from activation_research.metric_evaluator import MultiMetricHallucinationEvaluator

logger.remove()
logger.add(sys.stdout, level="INFO")

In [ ]:
# ---- Paths (edit these for your environment) ----
inference_json   = 'shared/goodwiki.zarr/generation.jsonl'
eval_json        = 'shared/goodwiki.zarr/eval_results.json'
activations_path = 'shared/goodwiki.zarr/activations.zarr'

# ---- Dataset parameters ----
backend          = 'zarr'           # 'zarr' or 'auto'
relevant_layers  = list(range(14, 30))
target_layers    = [22, 26]         # used for OOD evaluation
num_views        = 4
outlier_class    = 1                # halu=1 samples are the OOD class
fixed_layer      = None

# ---- Model / training hyperparams ----
device           = 'auto'
input_dim        = 4096
final_dim        = 512
recon_seq_len    = 64               # fixed decoder output length
recon_hidden_dim = 256
recon_lambda     = 1.0              # λ — weight of reconstruction term

max_epochs       = 50
batch_size       = 512
sub_batch_size   = 64
lr               = 1e-5
temperature      = 0.25

num_workers      = 16
persistent_workers = True

checkpoint_dir   = os.path.join('checkpoints', 'logprob_recon_contrastive')

## Load datasets

`include_response_logprobs=True` is **required** — the training function will raise
an error if logprobs are missing from the batch.

In [ ]:
ap = ActivationParser(
    inference_json=inference_json,
    eval_json=eval_json,
    activations_path=activations_path,
    verbose=False,
)

train_dataset = ap.get_dataset(
    'train',
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=num_views,
    backend=backend,
    include_response_logprobs=True,   # REQUIRED for logprob reconstruction
)
test_dataset = ap.get_dataset(
    'test',
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=num_views,
    backend=backend,
    include_response_logprobs=True,
)

print(f'train: {len(train_dataset)}')
print(f'test : {len(test_dataset)}')

In [ ]:
# ---- Validate that logprob data is actually present ----
# This catches the case where the zarr store was built without --log-response-logprobs.
_probe = train_dataset[0]
assert 'response_token_logprobs' in _probe, (
    "response_token_logprobs not found in dataset items. "
    "Re-run inference with --log-response-logprobs (or equivalent) to log per-token logprobs "
    "before training the logprob reconstruction model."
)

_lp = _probe['response_token_logprobs']
_mask = _probe.get('response_logprob_mask')
_n_valid = int(_mask.sum()) if _mask is not None else int((~_lp.isnan()).sum())
print(f'Logprob tensor shape : {_lp.shape}')
print(f'Valid (non-NaN) tokens: {_n_valid} / {_lp.shape[0]}')
print(f'Sample logprob range  : [{_lp[~_lp.isnan()].min():.3f}, {_lp[~_lp.isnan()].max():.3f}]')

# Also verify the collator picks up the key correctly.
_probe_batch = _contrastive_collate_with_logprob([train_dataset[i] for i in range(4)])
assert 'logprob' in _probe_batch, (
    "Collator did not produce a 'logprob' key. "
    "Check that _contrastive_collate_with_logprob recognises 'response_token_logprobs'."
)
print(f'Collated logprob shape: {_probe_batch["logprob"].shape}  (B, L)')
del _probe, _lp, _mask, _n_valid, _probe_batch

In [ ]:
# ---- Build model ----
model = LogprobReconProgressiveCompressor(
    input_dim=input_dim,
    final_dim=final_dim,
    input_dropout=0.3,
    recon_seq_len=recon_seq_len,
    recon_hidden_dim=recon_hidden_dim,
    recon_lambda=recon_lambda,
)
total_params = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
decoder_params = sum(p.numel() for p in model.decoder.parameters())
print(f'Total params  : {total_params:,}')
print(f'Encoder params: {encoder_params:,}')
print(f'Decoder params: {decoder_params:,}  (discarded at inference)')
model

In [ ]:
# ---- Train ----
train_contrastive_logprob_recon(
    model,
    train_dataset,
    test_dataset=test_dataset,
    epochs=max_epochs,
    batch_size=batch_size,
    sub_batch_size=sub_batch_size,
    lr=lr,
    temperature=temperature,
    device=device if device != 'auto' else ('cuda' if torch.cuda.is_available() else 'cpu'),
    num_workers=num_workers,
    persistent_workers=persistent_workers,
    use_labels=True,
    ignore_label=outlier_class,
    checkpoint_dir=checkpoint_dir,
    snapshot_every=10,
    snapshot_keep_last=5,
    recon_lambda=recon_lambda,
)

## OOD evaluation

`forward()` on `LogprobReconProgressiveCompressor` is identical to `ProgressiveCompressor`
— the decoder is not called.  All existing evaluation utilities work unchanged.

In [ ]:
eval_device = device if device != 'auto' else ('cuda' if torch.cuda.is_available() else 'cpu')

# Eval datasets — logprob not needed for evaluation.
train_dataset_for_inference = ap.get_dataset(
    'train', relevant_layers=target_layers, fixed_layer=fixed_layer, num_views=num_views, backend=backend,
)
eval_dataset = ap.get_dataset(
    'test', relevant_layers=target_layers, fixed_layer=fixed_layer, num_views=num_views, backend=backend,
)

train_loader_for_baseline = DataLoader(train_dataset_for_inference, batch_size=64, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

model_for_eval = model.to(eval_device)
model_for_eval.eval()

ood_eval = MultiMetricHallucinationEvaluator(
    activation_parser_df=ap.df,
    train_data_loader=train_loader_for_baseline,
    layers=None,
    batch_size=256,
    sub_batch_size=64,
    device=eval_device,
    num_workers=num_workers,
    persistent_workers=False,
    outlier_class=outlier_class,
    metrics=[
        'cosine',
        'mds',
        {
            'metric': 'knn',
            'kwargs': {
                'k': 50,
                'metric': 'euclidean',
                'calibrate_k': True,
                'k_candidates': [50, 100, 200, 500, 1000],
                'max_train_size': 200000,
                'sample_seed': 0,
            },
            'train_selection': 'all',
        },
    ],
)
ood_stats = ood_eval.compute(eval_loader, model_for_eval)
print('OOD metrics:', ood_stats)

## Evaluate across epoch snapshots

Load each saved snapshot, run OOD evaluation, and compare metrics across training epochs.
Also tracks the train-time reconstruction loss from each checkpoint to diagnose
how well the auxiliary decoder steered the encoder.

In [ ]:
snapshot_pattern = os.path.join(checkpoint_dir, 'contrastive_snapshot_epoch_*.pt')
snapshot_files = sorted(glob.glob(snapshot_pattern))

last_ckpt = os.path.join(checkpoint_dir, 'contrastive_last.pt')
if os.path.exists(last_ckpt):
    snapshot_files.append(last_ckpt)
snapshot_files = sorted(set(snapshot_files))

def _parse_epoch(path):
    m = re.search(r'epoch_(\d+)', os.path.basename(path))
    return f'epoch_{int(m.group(1))}' if m else 'last'

snapshot_info = [(p, _parse_epoch(p)) for p in snapshot_files]
print(f'Found {len(snapshot_info)} checkpoint(s):')
for p, label in snapshot_info:
    print(f'  {label:>12s}  ->  {p}')

In [ ]:
if 'train_loader_for_baseline' not in dir():
    train_loader_for_baseline = DataLoader(
        ap.get_dataset('train', relevant_layers=target_layers, fixed_layer=fixed_layer, num_views=num_views, backend=backend),
        batch_size=64, shuffle=False,
    )
if 'eval_loader' not in dir():
    eval_loader = DataLoader(
        ap.get_dataset('test', relevant_layers=target_layers, fixed_layer=fixed_layer, num_views=num_views, backend=backend),
        batch_size=64, shuffle=False,
    )

all_results = []

for ckpt_path, epoch_label in tqdm(snapshot_info, desc='Evaluating snapshots'):
    ckpt = torch.load(ckpt_path, map_location=eval_device)

    snapshot_model = LogprobReconProgressiveCompressor(
        input_dim=input_dim,
        final_dim=final_dim,
        input_dropout=0.3,
        recon_seq_len=recon_seq_len,
        recon_hidden_dim=recon_hidden_dim,
        recon_lambda=recon_lambda,
    )
    snapshot_model.load_state_dict(ckpt['model_state_dict'])
    snapshot_model = snapshot_model.to(eval_device)
    snapshot_model.eval()

    ood_eval = MultiMetricHallucinationEvaluator(
        activation_parser_df=ap.df,
        train_data_loader=train_loader_for_baseline,
        layers=None,
        batch_size=256,
        sub_batch_size=64,
        device=eval_device,
        num_workers=num_workers,
        persistent_workers=False,
        outlier_class=outlier_class,
        metrics=[
            'cosine',
            'mds',
            {
                'metric': 'knn',
                'kwargs': {
                    'k': 50,
                    'metric': 'euclidean',
                    'calibrate_k': True,
                    'k_candidates': [50, 100, 200, 500, 1000],
                    'max_train_size': 200000,
                    'sample_seed': 0,
                },
                'train_selection': 'all',
            },
        ],
    )
    stats = ood_eval.compute(eval_loader, snapshot_model)

    epoch_num = int(re.search(r'\d+', epoch_label).group()) if re.search(r'\d+', epoch_label) else 9999
    row = {
        'checkpoint': epoch_label,
        'epoch': epoch_num,
        'path': ckpt_path,
        # Training-time diagnostics from checkpoint
        'train_loss': ckpt.get('train_loss'),
        'train_supcon': ckpt.get('train_supcon'),
        'train_recon': ckpt.get('train_recon'),
        'recon_lambda': ckpt.get('recon_lambda', recon_lambda),
    }
    row.update(stats)
    all_results.append(row)
    logger.info(f'[{epoch_label}] supcon={ckpt.get("train_supcon", "?"):.4f} recon={ckpt.get("train_recon", "?"):.4f} | {stats}')

results_df = pd.DataFrame(all_results).sort_values('epoch').reset_index(drop=True)
print('\n=== Results across snapshots ===')
results_df

In [ ]:
import matplotlib.pyplot as plt

meta_cols = {'checkpoint', 'epoch', 'path', 'recon_lambda'}
metric_cols = [c for c in results_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(results_df[c])]

# Separate training diagnostics from OOD metrics for cleaner layout
train_diag_cols = [c for c in metric_cols if c.startswith('train_')]
ood_cols = [c for c in metric_cols if c not in train_diag_cols]

def _plot_group(cols, title):
    if not cols:
        return
    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4), squeeze=False)
    for ax, col in zip(axes.flatten(), cols):
        ax.plot(results_df['epoch'], results_df[col], marker='o', linewidth=1.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(col)
        ax.set_title(col)
        ax.grid(True, alpha=0.3)
    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

_plot_group(train_diag_cols, 'Training diagnostics vs Epoch')
_plot_group(ood_cols, 'OOD metrics vs Epoch')